# Fast Diffusion Illusions (Colab & Apple Silicon NPU)

Zero-shot & accelerated multi-view optical illusions based on [Diffusion-Illusions](https://github.com/RyannDaGreat/Diffusion-Illusions), sped up **27×–81×** (vs. 2,000-iter SD 1.4 SDS) and **137×–403×** (vs. 10,000-iter SDS) using modern open-weight few-step models — now with **auto-detected Apple Neural Engine (NPU)** acceleration via Core ML (`CPU_AND_NE`) when opened locally in Jupyter on Apple Silicon Macs.

| Task | Description | Fastest / Recommended Models | Typical Time (H100) |
| :--- | :--- | :--- | :--- |
| **`flip`** | 180° upright / upside-down ambigram | `sana_sprint` (2 steps, 1024px) / `flux2_klein` (4 steps, 512px) / `iris3b_turbo` (4 steps, 1024px) | **2.0s – 5.1s** |
| **`twisting_squares`** | 4×4 checkerboard ±90° tile rotation | `sana_sprint` (4 steps, 1024px) / `flux2_klein` (4 steps, 512px) | **2.4s – 2.8s** |
| **`rubiks_cube`** | 6 solved cube faces → 6 maximally dispersed scrambled faces (12 views, 18 moves, all 54 squares rotated 90°/180°/270°, max 2 tiles/face) | `flux2_klein` (6–8 steps, 576px) / `sana_sprint` (4 steps, 768px) / `iris3b_turbo` (4 steps, 768px) | **5.5s – 7.1s** |
| **`rotation_overlay`** | 2 translucent layers → 4 views at 0°/90°/180°/270° | `flux2_klein` (4–8 steps, 512px) / `sana_sprint` (2 steps, 1024px) | **3.0s – 4.2s** |
| **`hidden_overlay`** | 4 normal layers + 1 hidden `tanh(3·A·B·C·D)` view | `flux2_klein` (4 steps, 512px) / `iris3b_turbo` (4 steps, 1024px) | **3.1s – 6.1s** |
| **`apple_npu`** | Core ML `CPU_AND_NE` with ANE 4D `BC1S` attention + 6-bit palettized weights + `.mlmodelc` cache | `sd_turbo` / `sdxl_turbo` / `sana_sprint` (macOS 13+ Apple Silicon) | Local Apple NPU |
| **`fast_sds`** | Batched `bf16`/`fp16` + TAESD + `torch.compile` SD 1.5 SDS | Fallback optimization loop (350 iters, 512px) | **8.9s – 41.3s** |

> **Physical Exactness Guarantee:** All permutation tasks (`flip`, `twisting_squares`, `rubiks_cube`) and overlay tasks (`rotation_overlay`, `hidden_overlay`) compute the final views strictly via `op.derive(primes)` in pixel space (`MAE = 0.0`). Non-equivariant `±90°` tile rotations use step-wise pixel-space reconciliation so the VAE cannot "cheat" across tile boundaries.

**Hardware Auto-Detection Notes:**
- **Apple Silicon Mac (local Jupyter, macOS 13+):** Cell 1 automatically detects Apple Silicon, installs `coremltools>=8.0` + `scikit-learn`, and sets `BACKEND_MODE = "apple_npu"`. With `METHOD = "auto"` in Cell 4, models are exported once with ANE-friendly 4D `1×1 Conv2d` attention + 6-bit weight palettization, compiled/cached as `.mlmodelc`, and executed on the Apple NPU (`ComputeUnit.CPU_AND_NE`).
- **Colab NVIDIA GPUs:**
  - **T4 (15 GB):** Use `auto` (`flux2_klein` with auto CPU offload + chunked VAE), `sana_sprint`, or `fast_sds`.
  - **L4 (24 GB) / A100 / H100:** All GPU methods run fast; `iris3b_turbo` at `768px–1024px` shines on A100/L4/H100.

In [ ]:
#@title 1. Detect Accelerator (CUDA GPU / Apple NPU / MPS) & Install Dependencies
import platform, subprocess, sys, torch

has_cuda = torch.cuda.is_available()
has_mps = bool(hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
sys_name = platform.system()
machine = platform.machine()
is_apple_silicon = (sys_name == "Darwin") and (machine in ("arm64", "arm64e"))
mac_ver = platform.mac_ver()[0] if sys_name == "Darwin" else ""
mac_major = int(mac_ver.split(".")[0]) if (mac_ver and mac_ver.split(".")[0].isdigit()) else 0

if has_cuda:
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    cc = torch.cuda.get_device_capability(0)
    print(f"CUDA GPU: {props.name} | VRAM: {vram_gb:.1f} GB | Compute: {cc[0]}.{cc[1]} | BF16: {torch.cuda.is_bf16_supported()}")
elif is_apple_silicon:
    print(f"Apple Silicon detected ({machine}, macOS {mac_ver or 'unknown'}) | PyTorch MPS available: {has_mps}")
elif has_mps:
    print("MPS device detected.")
else:
    raise RuntimeError("No CUDA GPU or Apple Silicon accelerator detected! On Colab, go to Runtime -> Change runtime type -> GPU.")

INSTALL_IRIS3B = True  #@param {type:"boolean"}

pkgs = [
    "diffusers>=0.37.0",
    "transformers>=4.50.0",
    "accelerate>=1.3.0",
    "safetensors",
    "omegaconf",
    "huggingface_hub",
    "einops",
    "pillow",
]
if is_apple_silicon:
    pkgs.extend(["coremltools>=8.0", "scikit-learn"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs])
if INSTALL_IRIS3B and has_cuda:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "git+https://github.com/speridlabs/iris-3b.git"])

coreml_ver = None
if is_apple_silicon:
    try:
        import coremltools as ct
        coreml_ver = getattr(ct, "__version__", "unknown")
    except Exception as e:
        print("Warning: coremltools import failed:", e)

if has_cuda:
    BACKEND_MODE = "cuda"
elif is_apple_silicon and (mac_major == 0 or mac_major >= 13) and coreml_ver is not None:
    BACKEND_MODE = "apple_npu"
    print(f"Apple NPU enabled: Core ML {coreml_ver} (ComputeUnit.CPU_AND_NE)")
elif has_mps:
    BACKEND_MODE = "mps"
    print("Fallback: PyTorch MPS (Apple GPU only; Core ML NPU unavailable)")
else:
    BACKEND_MODE = "cpu"

print(f"Active BACKEND_MODE = {BACKEND_MODE!r} | Dependencies ready.")

In [ ]:
#@title 2. Core Illusion Operators & Prime Reconcilers
import gc
import itertools
import math
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
try:
    from IPython.display import display
except ImportError:
    display = lambda x: None


def tensor_to_pil(t: torch.Tensor) -> Image.Image:
    arr = (t.detach().float().clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255).round().astype("uint8")
    return Image.fromarray(arr)


def make_grid(pils: list[Image.Image]) -> Image.Image:
    w, h = pils[0].size
    n = len(pils)
    cols = 6 if n == 12 else n
    rows = (n + cols - 1) // cols
    grid = Image.new("RGB", (w * cols, h * rows))
    for i, im in enumerate(pils):
        r, c = divmod(i, cols)
        grid.paste(im, (c * w, r * h))
    return grid


class FastSSIM(nn.Module):
    def __init__(self, window_size: int = 11, channels: int = 3):
        super().__init__()
        sigma = 1.5
        coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g = g / g.sum()
        w2d = (g[:, None] @ g[None, :]).unsqueeze(0).unsqueeze(0)
        self.register_buffer("window", w2d.expand(channels, 1, window_size, window_size).contiguous())

    def forward(self, img1: torch.Tensor, img2: torch.Tensor) -> torch.Tensor:
        w = self.window.to(device=img1.device, dtype=img1.dtype)
        c = img1.shape[1]
        mu1 = F.conv2d(img1, w, padding=0, groups=c)
        mu2 = F.conv2d(img2, w, padding=0, groups=c)
        mu1_sq, mu2_sq, mu1_mu2 = mu1 * mu1, mu2 * mu2, mu1 * mu2
        sigma1_sq = F.conv2d(img1 * img1, w, padding=0, groups=c) - mu1_sq
        sigma2_sq = F.conv2d(img2 * img2, w, padding=0, groups=c) - mu2_sq
        sigma12 = F.conv2d(img1 * img2, w, padding=0, groups=c) - mu1_mu2
        c1, c2 = 0.01 ** 2, 0.03 ** 2
        return (((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / ((mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2))).mean()


class FastFourierImage(nn.Module):
    def __init__(self, height: int = 512, width: int = 512, num_channels: int = 3, hidden_dim: int = 256, num_features: int = 128, scale: float = 10.0):
        super().__init__()
        y = torch.linspace(0, 1, height + 1)[:-1]
        x = torch.linspace(0, 1, width + 1)[:-1]
        gy, gx = torch.meshgrid(y, x, indexing="ij")
        uv = torch.stack([gx, gy], dim=-1).reshape(-1, 2)
        proj = 2.0 * math.pi * (uv @ (torch.randn(2, num_features) * scale))
        feats = torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1).t().reshape(1, 2 * num_features, height, width).contiguous()
        self.register_buffer("features", feats)
        m = 2 * num_features
        self.net = nn.Sequential(
            nn.Conv2d(m, hidden_dim, 1), nn.ReLU(), nn.BatchNorm2d(hidden_dim),
            nn.Conv2d(hidden_dim, hidden_dim, 1), nn.ReLU(), nn.BatchNorm2d(hidden_dim),
            nn.Conv2d(hidden_dim, hidden_dim, 1), nn.ReLU(), nn.BatchNorm2d(hidden_dim),
            nn.Conv2d(hidden_dim, num_channels, 1), nn.Sigmoid(),
        )

    def forward(self) -> torch.Tensor:
        return self.net(self.features).squeeze(0)


class FlipOperator:
    num_primes = 1
    num_views = 2
    is_permutation = True
    equivariant_to_latent = True

    def derive(self, primes: list[torch.Tensor]) -> torch.Tensor:
        p = primes[0]
        return torch.stack([p, p.rot90(2, dims=(1, 2))], dim=0)

    def invert_and_average(self, views: torch.Tensor) -> torch.Tensor:
        p = 0.5 * (views[0] + views[1].rot90(-2, dims=(1, 2)))
        return self.derive([p])

    def invert_velocities(self, v: torch.Tensor) -> torch.Tensor:
        v0 = 0.5 * (v[0] + v[1].rot90(-2, dims=(1, 2)))
        return torch.stack([v0, v0.rot90(2, dims=(1, 2))], dim=0)


class TwistingSquaresOperator:
    num_primes = 1
    num_views = 2
    is_permutation = True
    equivariant_to_latent = False

    def __init__(self, num_divisions: int = 4):
        self.num_divisions = num_divisions

    def _twist(self, img: torch.Tensor, direction: int = 1) -> torch.Tensor:
        c, h, w = img.shape
        n = self.num_divisions
        th, tw = h // n, w // n
        tiles = img[:, : n * th, : n * tw].view(c, n, th, n, tw).permute(1, 3, 0, 2, 4).contiguous()
        out = torch.empty_like(tiles)
        for y in range(n):
            for x in range(n):
                k = direction if (x + y) % 2 == 0 else -direction
                out[y, x] = tiles[y, x].rot90(k, dims=(1, 2))
        return out.permute(2, 0, 3, 1, 4).reshape(c, n * th, n * tw).contiguous()

    def derive(self, primes: list[torch.Tensor]) -> torch.Tensor:
        p = primes[0]
        return torch.stack([p, self._twist(p, direction=1)], dim=0)

    def invert_and_average(self, views: torch.Tensor) -> torch.Tensor:
        p = 0.5 * (views[0] + self._twist(views[1], direction=-1))
        return self.derive([p])

    def invert_velocities(self, v: torch.Tensor) -> torch.Tensor:
        v0 = 0.5 * (v[0] + self._twist(v[1], direction=-1))
        return torch.stack([v0, self._twist(v0, direction=1)], dim=0)


class RotationOverlayOperator:
    num_primes = 2
    num_views = 4
    is_permutation = False
    equivariant_to_latent = False

    def __init__(self, brightness: float = 3.0):
        self.brightness = brightness

    def derive(self, primes: list[torch.Tensor]) -> torch.Tensor:
        bottom, top = primes
        return torch.stack([
            torch.tanh((bottom * top.rot90(k, dims=(1, 2)) * self.brightness).clamp(0.0, 99.0))
            for k in range(4)
        ], dim=0)

    @torch.enable_grad()
    def reconcile(self, targets: torch.Tensor, init_logits=None, steps: int = 60, lr: float = 0.35):
        targets = targets.detach().float().clamp(1e-4, 1.0 - 1e-4)
        if init_logits is None:
            b_logit = torch.zeros_like(targets[0], requires_grad=True)
            t_logit = torch.zeros_like(targets[0], requires_grad=True)
        else:
            b_logit = init_logits[0].detach().clone().requires_grad_(True)
            t_logit = init_logits[1].detach().clone().requires_grad_(True)
        opt = torch.optim.Adam([b_logit, t_logit], lr=lr)
        for _ in range(steps):
            opt.zero_grad(set_to_none=True)
            preds = self.derive([torch.sigmoid(b_logit), torch.sigmoid(t_logit)])
            F.mse_loss(preds, targets).backward()
            opt.step()
        with torch.no_grad():
            b, t = torch.sigmoid(b_logit), torch.sigmoid(t_logit)
            return self.derive([b, t]), [b, t], (b_logit.detach(), t_logit.detach())


class HiddenOverlayOperator:
    num_primes = 4
    num_views = 5
    is_permutation = False
    equivariant_to_latent = False

    def __init__(self, brightness: float = 3.0, hidden_weight: float = 3.0):
        self.brightness = brightness
        self.hidden_weight = hidden_weight

    def derive(self, primes: list[torch.Tensor]) -> torch.Tensor:
        a, b, c, d = primes
        z = torch.tanh((self.brightness * a * b * c * d).clamp(0.0, 99.0))
        return torch.stack([a, b, c, d, z], dim=0)

    def reconcile(self, targets: torch.Tensor, steps: int = 30, step_size: float = 0.015):
        t = targets.detach().float().clamp(1e-3, 1.0 - 1e-3)
        ta, tb, tc, td, tz = t[0], t[1], t[2], t[3], t[4]
        bl, lz = self.brightness, self.hidden_weight
        p = ta * tb * tc * td
        eps = tz - bl * p
        v_dot = (1.0 / (ta * ta)) + (1.0 / (tb * tb)) + (1.0 / (tc * tc)) + (1.0 / (td * td))
        scale = (bl * lz * p * eps) / (1.0 + (bl * bl) * lz * (p * p) * v_dot)
        a = (ta + scale / ta).clamp(0.0, 1.0)
        b = (tb + scale / tb).clamp(0.0, 1.0)
        c = (tc + scale / tc).clamp(0.0, 1.0)
        d = (td + scale / td).clamp(0.0, 1.0)
        for _ in range(steps):
            z = torch.tanh(bl * a * b * c * d)
            common = -2.0 * bl * lz * (tz - z) * (1.0 - z * z)
            ga = -2.0 * (ta - a) + common * (b * c * d)
            gb = -2.0 * (tb - b) + common * (a * c * d)
            gc = -2.0 * (tc - c) + common * (a * b * d)
            gd = -2.0 * (td - d) + common * (a * b * c)
            a = (a - step_size * ga).clamp(0.0, 1.0)
            b = (b - step_size * gb).clamp(0.0, 1.0)
            c = (c - step_size * gc).clamp(0.0, 1.0)
            d = (d - step_size * gd).clamp(0.0, 1.0)
        return self.derive([a, b, c, d]), [a, b, c, d]


class RubiksCubeOperator:
    """
    6 solved cube faces (U, L, F, R, B, D) -> 12 views: [6 before (solved), 6 after (scrambled)].
    Default scramble ("R' L' F2 U' D' R' B2 R B2 L' B2 L2 U2 L' U2 L U2 L'", 18 moves, all 54 squares rotated 90°/180°/270°) achieves the
    exact theoretical minimum dispersion across faces (max 2 stickers from any solved face per
    scrambled face; 18 shared same-face pairs vs 216 solved; 0/48 moving stickers on solved face; 0/54 keep original orientation).
    """
    num_primes = 6
    num_views = 12
    is_permutation = True
    equivariant_to_latent = False

    FACE_NAMES = ["U", "L", "F", "R", "B", "D"]
    FACE_FRAMES = {
        "U": ((0, 1, 0), (1, 0, 0), (0, 0, 1)),
        "L": ((-1, 0, 0), (0, 0, 1), (0, -1, 0)),
        "F": ((0, 0, 1), (1, 0, 0), (0, -1, 0)),
        "R": ((1, 0, 0), (0, 0, -1), (0, -1, 0)),
        "B": ((0, 0, -1), (-1, 0, 0), (0, -1, 0)),
        "D": ((0, -1, 0), (1, 0, 0), (0, 0, -1)),
    }
    DEFAULT_SCRAMBLE = "R' L' F2 U' D' R' B2 R B2 L' B2 L2 U2 L' U2 L U2 L'"

    def __init__(self, scramble: str | None = None):
        self.scramble = scramble or self.DEFAULT_SCRAMBLE
        self.moves = self.scramble.split()
        self.mapping = self._simulate_scramble(self.moves)
        fwd_idx = [0] * 54
        inv_idx = [0] * 54
        for dest_idx, (src_idx, k) in enumerate(self.mapping):
            fwd_idx[dest_idx] = k * 54 + src_idx
            inv_idx[src_idx] = ((4 - k) % 4) * 54 + dest_idx
        self._fwd_idx = fwd_idx
        self._inv_idx = inv_idx
        self._idx_cache: dict[tuple[torch.device, bool], torch.Tensor] = {}

    @classmethod
    def _dot(cls, a, b):
        return a[0] * b[0] + a[1] * b[1] + a[2] * b[2]

    @classmethod
    def _cross(cls, a, b):
        return (a[1] * b[2] - a[2] * b[1], a[2] * b[0] - a[0] * b[2], a[0] * b[1] - a[1] * b[0])

    @classmethod
    def _rot_cw(cls, v, ax):
        c = cls._cross(ax, v)
        d = cls._dot(ax, v)
        return (ax[0] * d - c[0], ax[1] * d - c[1], ax[2] * d - c[2])

    @classmethod
    def _simulate_scramble(cls, moves: list[str]) -> list[tuple[int, int]]:
        stickers = []
        for fi, fn in enumerate(cls.FACE_NAMES):
            n, u, v = cls.FACE_FRAMES[fn]
            for r in range(3):
                for c in range(3):
                    co, ro = c - 1, r - 1
                    pos = (n[0] + co * u[0] + ro * v[0], n[1] + co * u[1] + ro * v[1], n[2] + co * u[2] + ro * v[2])
                    stickers.append({"orig": fi * 9 + r * 3 + c, "pos": pos, "n": n, "u": u, "v": v})
        for mv in moves:
            ax = cls.FACE_FRAMES[mv[0]][0]
            times = 1 if len(mv) == 1 else (2 if mv[1] == "2" else 3)
            for s in stickers:
                if cls._dot(s["pos"], ax) == 1:
                    for _ in range(times):
                        s["pos"] = cls._rot_cw(s["pos"], ax)
                        s["n"] = cls._rot_cw(s["n"], ax)
                        s["u"] = cls._rot_cw(s["u"], ax)
                        s["v"] = cls._rot_cw(s["v"], ax)
        norm_to_f = {cls.FACE_FRAMES[fn][0]: fi for fi, fn in enumerate(cls.FACE_NAMES)}
        mapping = [None] * 54
        for s in stickers:
            fd = norm_to_f[s["n"]]
            _, Ud, Vd = cls.FACE_FRAMES[cls.FACE_NAMES[fd]]
            cd = cls._dot(s["pos"], Ud) + 1
            rd = cls._dot(s["pos"], Vd) + 1
            dest_idx = fd * 9 + rd * 3 + cd
            du, dv = cls._dot(s["u"], Ud), cls._dot(s["u"], Vd)
            k = {(1, 0): 0, (0, -1): 1, (-1, 0): 2, (0, 1): 3}[(du, dv)]
            mapping[dest_idx] = (s["orig"], k)
        return mapping

    def confusion_matrix(self) -> list[list[int]]:
        N = [[0] * 6 for _ in range(6)]
        for dest_idx, (src_idx, _) in enumerate(self.mapping):
            N[dest_idx // 9][src_idx // 9] += 1
        return N

    def _get_idx(self, device: torch.device, inverse: bool = False) -> torch.Tensor:
        key = (device, inverse)
        if key not in self._idx_cache:
            arr = self._inv_idx if inverse else self._fwd_idx
            self._idx_cache[key] = torch.tensor(arr, dtype=torch.long, device=device)
        return self._idx_cache[key]

    def _permute_faces(self, faces: torch.Tensor, inverse: bool = False) -> torch.Tensor:
        _, c, h, w = faces.shape
        th, tw = h // 3, w // 3
        hc, wc = 3 * th, 3 * tw
        sub = faces[:, :, :hc, :wc]
        tiles = sub.view(6, c, 3, th, 3, tw).permute(0, 2, 4, 1, 3, 5).reshape(54, c, th, tw)
        variants = torch.cat([tiles, tiles.rot90(1, dims=(2, 3)), tiles.rot90(2, dims=(2, 3)), tiles.rot90(3, dims=(2, 3))], dim=0)
        out_tiles = variants[self._get_idx(faces.device, inverse=inverse)]
        out_sub = out_tiles.view(6, 3, 3, c, th, tw).permute(0, 3, 1, 4, 2, 5).reshape(6, c, hc, wc)
        if hc == h and wc == w:
            return out_sub
        out = faces.clone()
        out[:, :, :hc, :wc] = out_sub
        return out

    def derive(self, primes: list[torch.Tensor]) -> torch.Tensor:
        before = torch.stack(primes, dim=0)
        after = self._permute_faces(before, inverse=False)
        return torch.cat([before, after], dim=0)

    def invert_and_average(self, views: torch.Tensor) -> torch.Tensor:
        before_in, after_in = views[:6], views[6:]
        before_rec = 0.5 * (before_in + self._permute_faces(after_in, inverse=True))
        return self.derive([before_rec[i] for i in range(6)])

    def invert_velocities(self, v: torch.Tensor) -> torch.Tensor:
        v_before = 0.5 * (v[:6] + self._permute_faces(v[6:], inverse=True))
        return torch.cat([v_before, self._permute_faces(v_before, inverse=False)], dim=0)

    def match_scrambled_faces(self, views: torch.Tensor) -> tuple[list[int], float, float]:
        solved_scrambled = self._permute_faces(views[:6].detach().float(), inverse=False)
        cand = views[6:12].detach().float()
        a = solved_scrambled.reshape(6, -1)
        b = cand.reshape(6, -1)
        n = float(a.shape[1])
        a2 = (a * a).sum(dim=1, keepdim=True)
        b2 = (b * b).sum(dim=1, keepdim=True).t()
        cost = ((a2 + b2 - 2.0 * (a @ b.t())) / n).cpu().tolist()
        identity_cost = sum(cost[j][j] for j in range(6)) / 6.0
        best_cost = float("inf")
        best_perm = list(range(6))
        for p in itertools.permutations(range(6)):
            c = cost[0][p[0]] + cost[1][p[1]] + cost[2][p[2]] + cost[3][p[3]] + cost[4][p[4]] + cost[5][p[5]]
            if c < best_cost:
                best_cost = c
                best_perm = list(p)
        return best_perm, float(identity_cost), float(best_cost / 6.0)


def get_operator(task: str):
    if task == "flip":
        return FlipOperator()
    if task == "twisting_squares":
        return TwistingSquaresOperator(num_divisions=4)
    if task == "rubiks_cube":
        return RubiksCubeOperator()
    if task == "rotation_overlay":
        return RotationOverlayOperator(brightness=3.0)
    if task == "hidden_overlay":
        return HiddenOverlayOperator(brightness=3.0, hidden_weight=3.0)
    raise ValueError(f"Unknown task: {task}")

print("Operators ready.")


In [ ]:
#@title 3. Fast Generation Runners (CUDA Auto-Offload, Apple NPU Core ML, & Model Caching)
from contextlib import nullcontext

_PIPE_CACHE = {}


def _get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def _sync_device(device: torch.device | None = None):
    d = device or _get_device()
    if d.type == "cuda":
        torch.cuda.synchronize()
    elif d.type == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "synchronize"):
        torch.mps.synchronize()


def _empty_cache(device: torch.device | None = None):
    d = device or _get_device()
    if d.type == "cuda" and torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif d.type == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()


def _clear_other_pipes(keep_key: str):
    for k in list(_PIPE_CACHE.keys()):
        if k != keep_key:
            del _PIPE_CACHE[k]
    gc.collect()
    _empty_cache()


def _best_dtype(device: torch.device | None = None):
    d = device or _get_device()
    if d.type == "cuda":
        if torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16
    if d.type == "mps":
        return torch.float16
    return torch.float32


def _low_vram_mode(device: torch.device | None = None) -> bool:
    d = device or _get_device()
    if d.type != "cuda":
        return False
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    return vram_gb < 22.0




def _reorder_rubiks_scrambled(op, imgs: torch.Tensor, prompts: list[str], *extras):
    if not isinstance(op, RubiksCubeOperator):
        return None
    perm, mse_before, mse_after = op.match_scrambled_faces(imgs)
    idx_cpu = torch.tensor(perm, dtype=torch.long)
    imgs[6:12] = imgs[6:12][idx_cpu.to(imgs.device)]
    for obj in extras:
        if obj is None:
            continue
        if isinstance(obj, np.ndarray):
            if obj.shape[0] >= 12:
                obj[6:12] = obj[6:12][perm]
        elif isinstance(obj, torch.Tensor):
            obj[6:12] = obj[6:12][idx_cpu.to(obj.device)]
    mapping = {fn: prompts[6 + p] for fn, p in zip(op.FACE_NAMES, perm)}
    print(f"Rubik's step-0 scrambled face match: perm={perm} | MSE {mse_before:.5f} -> {mse_after:.5f} | {mapping}")
    return mapping


def run_flux2_klein(task: str, prompts: list[str], num_steps: int = 4, size: int = 512, seed: int = 42, reconcile_steps: int = 60):
    from diffusers import Flux2KleinPipeline
    from diffusers.pipelines.flux2.pipeline_flux2_klein import compute_empirical_mu, retrieve_timesteps

    op = get_operator(task)
    assert len(prompts) == op.num_views
    pixel_reconcile = (not op.is_permutation) or (not op.equivariant_to_latent)

    device = _get_device()
    dtype = _best_dtype(device)
    low_vram = _low_vram_mode(device)

    if "flux2_klein" not in _PIPE_CACHE:
        _clear_other_pipes("flux2_klein")
        t_l = time.perf_counter()
        pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B", torch_dtype=dtype)
        if not low_vram:
            pipe = pipe.to(device)
        _PIPE_CACHE["flux2_klein"] = pipe
        print(f"Loaded FLUX.2-klein-4B in {time.perf_counter() - t_l:.1f}s (low_vram={low_vram}, dtype={dtype})")
    pipe = _PIPE_CACHE["flux2_klein"]

    _sync_device(device)
    t0 = time.perf_counter()
    gen = torch.Generator(device=device).manual_seed(seed)

    vae_chunk = 2 if low_vram else 4
    def _vae_dec(z):
        if z.shape[0] <= vae_chunk:
            return pipe.vae.decode(z, return_dict=False)[0]
        return torch.cat([pipe.vae.decode(z[k:k + vae_chunk], return_dict=False)[0] for k in range(0, z.shape[0], vae_chunk)], dim=0)

    def _vae_enc(x_in):
        if x_in.shape[0] <= vae_chunk:
            return pipe.vae.encode(x_in).latent_dist.mean
        return torch.cat([pipe.vae.encode(x_in[k:k + vae_chunk]).latent_dist.mean for k in range(0, x_in.shape[0], vae_chunk)], dim=0)

    with torch.no_grad():
        if low_vram:
            pipe.transformer.to("cpu")
            pipe.vae.to("cpu")
            _empty_cache(device)
            pipe.text_encoder.to(device)

        prompt_embeds, text_ids = pipe.encode_prompt(
            prompt=prompts, device=device, num_images_per_prompt=1, max_sequence_length=512
        )

        if low_vram:
            pipe.text_encoder.to("cpu")
            _empty_cache(device)
            pipe.transformer.to(device)
            pipe.vae.to(device)

        num_channels_latents = pipe.transformer.config.in_channels // 4
        latent_height = 2 * (int(size) // (pipe.vae_scale_factor * 2))
        latent_width = 2 * (int(size) // (pipe.vae_scale_factor * 2))
        pack_h, pack_w = latent_height // 2, latent_width // 2

        latents, latent_ids = pipe.prepare_latents(
            batch_size=op.num_views, num_latents_channels=num_channels_latents,
            height=size, width=size, dtype=prompt_embeds.dtype, device=device, generator=gen, latents=None,
        )

        if op.is_permutation and not pixel_reconcile:
            spatial = pipe._unpack_latents_with_ids(latents[:op.num_primes], latent_ids[:op.num_primes], pack_h, pack_w)
            unpatch = pipe._unpatchify_latents(spatial)
            latents = pipe._pack_latents(pipe._patchify_latents(op.derive([unpatch[k] for k in range(op.num_primes)]))).to(dtype)

        sigmas = np.linspace(1.0, 1.0 / num_steps, num_steps)
        mu = compute_empirical_mu(image_seq_len=latents.shape[1], num_steps=num_steps)
        timesteps, _ = retrieve_timesteps(pipe.scheduler, num_steps, device, sigmas=sigmas, mu=mu)
        sched_sigmas = pipe.scheduler.sigmas.to(device=device, dtype=torch.float32)

        bn_mean = pipe.vae.bn.running_mean.view(1, -1, 1, 1).to(device, torch.float32)
        bn_std = torch.sqrt(pipe.vae.bn.running_var.view(1, -1, 1, 1) + pipe.vae.config.batch_norm_eps).to(device, torch.float32)

        def _packed_to_vae(packed):
            sp = pipe._unpack_latents_with_ids(packed, latent_ids, pack_h, pack_w).float()
            return pipe._unpatchify_latents(sp * bn_std + bn_mean)

        def _vae_to_packed(vae_lat):
            patch_norm = (pipe._patchify_latents(vae_lat.float()) - bn_mean) / bn_std
            return pipe._pack_latents(patch_norm)

        warm_logits, primes, last_rec_01 = None, None, None
        tx_chunk = 4 if (low_vram and latents.shape[0] > 4) else (6 if (latents.shape[0] > 6 and size >= 576) else latents.shape[0])
        for i, t in enumerate(timesteps):
            if tx_chunk < latents.shape[0]:
                outs = []
                for s0 in range(0, latents.shape[0], tx_chunk):
                    s1 = s0 + tx_chunk
                    outs.append(
                        pipe.transformer(
                            hidden_states=latents[s0:s1], timestep=t.expand(s1 - s0).to(latents.dtype) / 1000.0,
                            guidance=None, encoder_hidden_states=prompt_embeds[s0:s1],
                            txt_ids=text_ids[s0:s1], img_ids=latent_ids[s0:s1], return_dict=False,
                        )[0][:, : latents.size(1)]
                    )
                noise_pred = torch.cat(outs, dim=0)
            else:
                timestep = t.expand(latents.shape[0]).to(latents.dtype)
                noise_pred = pipe.transformer(
                    hidden_states=latents, timestep=timestep / 1000.0, guidance=None,
                    encoder_hidden_states=prompt_embeds, txt_ids=text_ids, img_ids=latent_ids, return_dict=False,
                )[0][:, : latents.size(1)]

            sigma, sigma_next = sched_sigmas[i], sched_sigmas[i + 1]
            x0_packed = latents.float() - sigma * noise_pred.float()

            if op.is_permutation and not pixel_reconcile:
                x0_lat_rec = op.invert_and_average(_packed_to_vae(x0_packed))
                v_eff = ((latents.float() - _vae_to_packed(x0_lat_rec)) / sigma.clamp(min=1e-4)).to(dtype)
                latents = pipe.scheduler.step(v_eff, t, latents, return_dict=False)[0]
            else:
                x0_lat = _packed_to_vae(x0_packed).to(pipe.vae.dtype)
                imgs_01 = ((_vae_dec(x0_lat).float() + 1.0) * 0.5).clamp(0.0, 1.0)
                if i == 0 and isinstance(op, RubiksCubeOperator):
                    _reorder_rubiks_scrambled(op, imgs_01, prompts, latents, prompt_embeds, text_ids, latent_ids)
                if op.is_permutation:
                    rec_01 = op.invert_and_average(imgs_01)
                    primes = [rec_01[k] for k in range(op.num_primes)]
                elif isinstance(op, RotationOverlayOperator):
                    rec_01, primes, warm_logits = op.reconcile(imgs_01, init_logits=warm_logits, steps=reconcile_steps)
                else:
                    rec_01, primes = op.reconcile(imgs_01)
                last_rec_01 = rec_01
                if i + 1 < len(timesteps):
                    rec_lat = _vae_enc((2.0 * rec_01 - 1.0).to(pipe.vae.dtype))
                    v_rec = (latents.float() - _vae_to_packed(rec_lat)) / sigma.clamp(min=1e-4)
                    latents = (latents.float() + (sigma_next - sigma) * v_rec).to(dtype)

        if op.is_permutation and not pixel_reconcile:
            imgs = _vae_dec(_packed_to_vae(latents).to(pipe.vae.dtype))
            decoded_01 = ((imgs.float() + 1.0) * 0.5).clamp(0.0, 1.0)
            primes = [decoded_01[k] for k in range(op.num_primes)]
        elif op.is_permutation:
            primes = [last_rec_01[k] for k in range(op.num_primes)]
        views_01 = op.derive(primes)

    _sync_device(device)
    return primes, views_01, time.perf_counter() - t0


def run_sana_sprint(task: str, prompts: list[str], num_steps: int = 2, size: int = 1024, seed: int = 42, reconcile_steps: int = 60):
    from diffusers import SanaSprintPipeline
    from diffusers.pipelines.sana.pipeline_sana_sprint import retrieve_timesteps

    op = get_operator(task)
    assert len(prompts) == op.num_views
    pixel_reconcile = (not op.is_permutation) or (not op.equivariant_to_latent)

    device = _get_device()
    dtype = _best_dtype(device)
    low_vram = _low_vram_mode(device)

    if "sana_sprint" not in _PIPE_CACHE:
        _clear_other_pipes("sana_sprint")
        t_l = time.perf_counter()
        pipe = SanaSprintPipeline.from_pretrained("Efficient-Large-Model/Sana_Sprint_1.6B_1024px_diffusers", torch_dtype=dtype)
        pipe.vae.to(dtype=torch.float32)  # Prevent DC-AE overflow
        if not low_vram:
            pipe = pipe.to(device)
        _PIPE_CACHE["sana_sprint"] = pipe
        print(f"Loaded SANA-Sprint 1.6B in {time.perf_counter() - t_l:.1f}s (low_vram={low_vram}, dtype={dtype})")
    pipe = _PIPE_CACHE["sana_sprint"]

    _sync_device(device)
    t0 = time.perf_counter()
    gen = torch.Generator(device=device).manual_seed(seed)

    vae_chunk = 2 if low_vram else 4
    def _vae_dec(z):
        if z.shape[0] <= vae_chunk:
            return pipe.vae.decode(z, return_dict=False)[0]
        return torch.cat([pipe.vae.decode(z[k:k + vae_chunk], return_dict=False)[0] for k in range(0, z.shape[0], vae_chunk)], dim=0)

    def _vae_enc(x_in):
        if x_in.shape[0] <= vae_chunk:
            return pipe.vae.encode(x_in).latent
        return torch.cat([pipe.vae.encode(x_in[k:k + vae_chunk]).latent for k in range(0, x_in.shape[0], vae_chunk)], dim=0)

    with torch.no_grad():
        if low_vram:
            pipe.transformer.to("cpu")
            pipe.vae.to("cpu")
            _empty_cache(device)
            pipe.text_encoder.to(device)

        prompt_embeds, prompt_attention_mask = pipe.encode_prompt(
            prompts, num_images_per_prompt=1, device=device, clean_caption=False, max_sequence_length=300
        )

        if low_vram:
            pipe.text_encoder.to("cpu")
            _empty_cache(device)
            pipe.transformer.to(device)
            pipe.vae.to(device)

        timesteps, _ = retrieve_timesteps(
            pipe.scheduler, num_steps, device, timesteps=None, sigmas=None,
            max_timesteps=1.57080, intermediate_timesteps=1.3 if num_steps == 2 else None,
        )
        if hasattr(pipe.scheduler, "set_begin_index"):
            pipe.scheduler.set_begin_index(0)

        latents = pipe.prepare_latents(op.num_views, pipe.transformer.config.in_channels, size, size, torch.float32, device, gen, None)
        if op.is_permutation and not pixel_reconcile:
            latents = op.derive([latents[k] for k in range(op.num_primes)])
        sigma_data = pipe.scheduler.config.sigma_data
        latents = latents * sigma_data

        guidance = torch.full([op.num_views], 4.5, device=device, dtype=torch.float32).to(prompt_embeds.dtype)
        guidance = guidance * pipe.transformer.config.guidance_embeds_scale

        timesteps_loop = timesteps[:-1]
        warm_logits, primes, denoised, last_rec_01 = None, None, None, None
        tx_chunk = 4 if (low_vram and latents.shape[0] > 4) else (6 if (latents.shape[0] > 6 and size >= 768) else latents.shape[0])
        for i, t in enumerate(timesteps_loop):
            timestep = t.expand(latents.shape[0])
            scm_t = torch.sin(timestep) / (torch.cos(timestep) + torch.sin(timestep))
            scm_exp = scm_t.view(-1, 1, 1, 1)
            latent_in = (latents / sigma_data) * torch.sqrt(scm_exp ** 2 + (1.0 - scm_exp) ** 2)

            if tx_chunk < latents.shape[0]:
                outs = []
                for s0 in range(0, latents.shape[0], tx_chunk):
                    s1 = s0 + tx_chunk
                    outs.append(
                        pipe.transformer(
                            latent_in[s0:s1].to(dtype=dtype), encoder_hidden_states=prompt_embeds[s0:s1].to(dtype=dtype),
                            encoder_attention_mask=prompt_attention_mask[s0:s1], guidance=guidance[s0:s1],
                            timestep=scm_t[s0:s1], return_dict=False,
                        )[0].float()
                    )
                noise_pred = torch.cat(outs, dim=0)
            else:
                noise_pred = pipe.transformer(
                    latent_in.to(dtype=dtype), encoder_hidden_states=prompt_embeds.to(dtype=dtype),
                    encoder_attention_mask=prompt_attention_mask, guidance=guidance, timestep=scm_t, return_dict=False,
                )[0].float()
            noise_pred = ((1.0 - 2.0 * scm_exp) * latent_in + (1.0 - 2.0 * scm_exp + 2.0 * scm_exp ** 2) * noise_pred) / torch.sqrt(scm_exp ** 2 + (1.0 - scm_exp) ** 2)
            noise_pred = noise_pred * sigma_data

            if op.is_permutation and not pixel_reconcile:
                noise_pred = op.invert_velocities(noise_pred)
                latents, denoised = pipe.scheduler.step(noise_pred, timestep, latents, return_dict=False)
            else:
                _, denoised_step = pipe.scheduler.step(noise_pred, timestep, latents, return_dict=False)
                x0_lat = (denoised_step / sigma_data).to(pipe.vae.dtype)
                imgs_01 = ((_vae_dec(x0_lat / pipe.vae.config.scaling_factor).float() + 1.0) * 0.5).clamp(0.0, 1.0)
                if i == 0 and isinstance(op, RubiksCubeOperator):
                    _reorder_rubiks_scrambled(op, imgs_01, prompts, latents, prompt_embeds, prompt_attention_mask)
                if op.is_permutation:
                    rec_01 = op.invert_and_average(imgs_01)
                    primes = [rec_01[k] for k in range(op.num_primes)]
                elif isinstance(op, RotationOverlayOperator):
                    rec_01, primes, warm_logits = op.reconcile(imgs_01, init_logits=warm_logits, steps=reconcile_steps)
                else:
                    rec_01, primes = op.reconcile(imgs_01)
                last_rec_01 = rec_01
                if i + 1 < len(timesteps_loop):
                    rec_lat = _vae_enc((2.0 * rec_01 - 1.0).to(pipe.vae.dtype)) * pipe.vae.config.scaling_factor
                    rec_denoised = rec_lat.float() * sigma_data
                    t_next = timesteps_loop[i + 1]
                    latents = math.cos(float(t_next)) * rec_denoised + math.sin(float(t_next)) * torch.randn_like(rec_denoised) * sigma_data
                    denoised = rec_denoised

        if op.is_permutation and not pixel_reconcile:
            x0_lat = (denoised / sigma_data).to(pipe.vae.dtype)
            imgs = _vae_dec(x0_lat / pipe.vae.config.scaling_factor)
            decoded_01 = ((imgs.float() + 1.0) * 0.5).clamp(0.0, 1.0)
            primes = [decoded_01[k] for k in range(op.num_primes)]
        elif op.is_permutation:
            primes = [last_rec_01[k] for k in range(op.num_primes)]
        views_01 = op.derive(primes)

    _sync_device(device)
    return primes, views_01, time.perf_counter() - t0


def run_iris3b_turbo(task: str, prompts: list[str], size: int = 1024, seed: int = 42, reconcile_steps: int = 60):
    from huggingface_hub import snapshot_download
    from omegaconf import OmegaConf
    from iris3b.config import inference_config
    from iris3b.models.dit import IrisDiT
    from iris3b.registry import TEXT_ENCODERS
    from iris3b.sampling import load_for_inference
    import iris3b.text  # noqa: F401

    op = get_operator(task)
    assert len(prompts) == op.num_views
    device = _get_device()
    dtype = _best_dtype(device)
    low_vram = _low_vram_mode(device)

    if "iris3b" not in _PIPE_CACHE:
        _clear_other_pipes("iris3b")
        t_l = time.perf_counter()
        ckpt_dir = Path(snapshot_download("speridlabs/iris-3b", allow_patterns=["turbo/*"])) / "turbo"
        raw, weights = load_for_inference(ckpt_dir)
        cfg = inference_config(raw, [])
        sigmas = list(OmegaConf.load(ckpt_dir / "distill.yaml").dmd.sigmas)
        with torch.device("meta"):
            model = IrisDiT(cfg.model)
        model.load_state_dict(weights, strict=True, assign=True)
        model = model.eval().to(device=device, dtype=dtype)
        text_encoder = TEXT_ENCODERS.build(cfg.text_encoder.name, cfg.text_encoder, device=device)
        _PIPE_CACHE["iris3b"] = (model, text_encoder, cfg, sigmas)
        print(f"Loaded Iris-3B Turbo in {time.perf_counter() - t_l:.1f}s (dtype={dtype})")
    model, text_encoder, cfg, sigmas = _PIPE_CACHE["iris3b"]

    _sync_device(device)
    t0 = time.perf_counter()
    gen = torch.Generator(device=device).manual_seed(seed)

    with torch.no_grad(), (torch.autocast(device.type, dtype=dtype) if device.type in ("cuda", "mps") else nullcontext()):
        encoding = text_encoder.encode(prompts)
        cond = encoding.embeddings.to(device=device, dtype=dtype)
        mask = encoding.mask.to(device=device)

        if op.is_permutation:
            base = torch.randn(op.num_primes, model.cfg.in_channels, size, size, generator=gen, device=device)
            x = op.derive([base[k] for k in range(op.num_primes)])
        else:
            x = torch.randn(op.num_views, model.cfg.in_channels, size, size, generator=gen, device=device)

        warm_logits, primes = None, None
        sigma = torch.full((x.shape[0], 1, 1, 1), sigmas[0], device=device, dtype=torch.float32)
        chunk = 4 if (low_vram and x.shape[0] > 4) else (6 if (x.shape[0] > 6 and size >= 576) else x.shape[0])
        for i in range(len(sigmas)):
            if chunk < x.shape[0]:
                outs = []
                t_all = sigma.flatten() * cfg.flow.num_train_timesteps
                for s0 in range(0, x.shape[0], chunk):
                    s1 = s0 + chunk
                    outs.append(model(x[s0:s1].to(dtype), t_all[s0:s1], cond[s0:s1], y_mask=mask[s0:s1]).x.float())
                out = torch.cat(outs, dim=0)
            else:
                out = model(x.to(dtype), sigma.flatten() * cfg.flow.num_train_timesteps, cond, y_mask=mask).x.float()
            clean = out if cfg.flow.prediction == "x" else x - sigma * out
            if i == 0 and isinstance(op, RubiksCubeOperator):
                _reorder_rubiks_scrambled(op, ((clean + 1.0) * 0.5).clamp(0.0, 1.0), prompts, clean, x, cond, mask)
            if op.is_permutation:
                clean_rec = op.invert_and_average(clean)
            else:
                clean_01 = ((clean + 1.0) * 0.5).clamp(0.0, 1.0)
                if isinstance(op, RotationOverlayOperator):
                    clean_rec_01, primes, warm_logits = op.reconcile(clean_01, init_logits=warm_logits, steps=reconcile_steps)
                else:
                    clean_rec_01, primes = op.reconcile(clean_01)
                clean_rec = 2.0 * clean_rec_01 - 1.0
            if i + 1 < len(sigmas):
                implied_noise = (x - (1.0 - sigma) * clean_rec) / sigma
                sigma = torch.full_like(sigma, sigmas[i + 1])
                x = (1.0 - sigma) * clean_rec + sigma * implied_noise

    if op.is_permutation:
        raw_01 = ((clean_rec + 1.0) * 0.5).clamp(0.0, 1.0)
        primes = [raw_01[k] for k in range(op.num_primes)]
    views_01 = op.derive(primes)
    _sync_device(device)
    return primes, views_01, time.perf_counter() - t0


def run_fast_sds(task: str, prompts: list[str], sds_iters: int = 350, size: int = 512, seed: int = 42, use_compile: bool | None = None):
    from diffusers import StableDiffusionPipeline, DDIMScheduler, AutoencoderTiny

    torch.manual_seed(seed)
    device = _get_device()
    dtype = _best_dtype(device)
    if use_compile is None:
        use_compile = (device.type == "cuda") and (torch.cuda.get_device_capability(0)[0] >= 8)

    op = get_operator(task)
    assert len(prompts) == op.num_views

    if "fast_sds" not in _PIPE_CACHE:
        _clear_other_pipes("fast_sds")
        t_l = time.perf_counter()
        pipe = StableDiffusionPipeline.from_pretrained(
            "stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=dtype, safety_checker=None, requires_safety_checker=False,
        ).to(device)
        scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
        taesd = AutoencoderTiny.from_pretrained("madebyollin/taesd", torch_dtype=dtype).to(device).eval()
        for p in [*pipe.unet.parameters(), *taesd.parameters()]:
            p.requires_grad_(False)
        unet = torch.compile(pipe.unet.eval(), mode="reduce-overhead") if use_compile else pipe.unet.eval()
        _PIPE_CACHE["fast_sds"] = (pipe, unet, taesd, scheduler.alphas_cumprod.to(device=device, dtype=torch.float32))
        print(f"Loaded SD 1.5 + TAESD in {time.perf_counter() - t_l:.1f}s (compile={use_compile}, dtype={dtype})")
    pipe, unet, taesd, alphas_cumprod = _PIPE_CACHE["fast_sds"]

    with torch.no_grad():
        conds, unconds = zip(*[
            pipe.encode_prompt(prompt=p_txt, device=device, num_images_per_prompt=1, do_classifier_free_guidance=True, negative_prompt="")
            for p_txt in prompts
        ])
        cfg_embs = torch.cat([torch.cat(unconds, dim=0), torch.cat(conds, dim=0)], dim=0)

    primes_mod = nn.ModuleList([FastFourierImage(height=size, width=size).to(device) for _ in range(op.num_primes)])
    optim = torch.optim.AdamW(primes_mod.parameters(), lr=5e-4)

    _sync_device(device)
    t0 = time.perf_counter()
    for it in range(sds_iters):
        optim.zero_grad(set_to_none=True)
        frac = it / max(1, sds_iters - 1)
        t_hi, t_lo = int(960 - 460 * frac), int(300 - 280 * frac)
        t = torch.randint(t_lo, max(t_lo + 1, t_hi), (1,), device=device, dtype=torch.long)

        views = op.derive([p() for p in primes_mod])
        views_512 = F.interpolate(views, size=(512, 512), mode="bilinear", align_corners=False) if size != 512 else views
        with (torch.autocast(device.type, dtype=dtype) if device.type in ("cuda", "mps") else nullcontext()):
            latents = taesd.encode(2.0 * views_512 - 1.0).latents

        with torch.no_grad():
            noise = torch.randn_like(latents, dtype=torch.float32)
            a_t = alphas_cumprod[t]
            latents_noisy = (a_t.sqrt().view(1, 1, 1, 1) * latents.float() + (1.0 - a_t).sqrt().view(1, 1, 1, 1) * noise).to(dtype)
            n_uncond, n_cond = unet(torch.cat([latents_noisy, latents_noisy], dim=0), t, encoder_hidden_states=cfg_embs).sample.float().chunk(2, dim=0)
            n_cfg = n_uncond + 50.0 * (n_cond - n_uncond)
            grad = (1.0 - a_t).view(1, 1, 1, 1) * (n_cfg - noise) * (0.1 / op.num_views)

        latents.backward(gradient=grad.to(latents.dtype))
        optim.step()

    with torch.no_grad():
        primes = [p() for p in primes_mod]
        views_01 = op.derive(primes)
    _sync_device(device)
    return primes, views_01, time.perf_counter() - t0



def run_apple_npu(
    task: str,
    prompts: list[str],
    model_key: str = "sd_turbo",
    num_steps: int = 4,
    size: int | None = None,
    quantize_nbits: int | None = 6,
    compute_units: str = "CPU_AND_NE",
    seed: int = 42,
    reconcile_steps: int = 50,
):
    """Run zero-shot illusion generation on the Apple Neural Engine (Core ML CPU_AND_NE)."""
    import json, platform, tempfile

    # 1. If running from the cloned ambigram repository, delegate directly to fast_illusions.apple_npu
    try:
        from fast_illusions.apple_npu import apple_npu_info, run_apple_npu_zeroshot
        _use_repo = True
    except ImportError:
        _use_repo = False

    op = get_operator(task)
    assert len(prompts) == op.num_views, f"Expected {op.num_views} prompts for {task}, got {len(prompts)}"
    is_rubiks = task == "rubiks_cube"
    if size is None:
        size = (768 if is_rubiks else 1024) if model_key == "sana_sprint" else ((576 if is_rubiks else 512) if model_key == "sdxl_turbo" else (480 if is_rubiks else 512))

    if _use_repo:
        t_dir = tempfile.mkdtemp(prefix="apple_npu_")
        metrics = run_apple_npu_zeroshot(
            task=task,
            prompts=prompts,
            model_key=model_key,
            coreml_dir=f"./coreml_models/{model_key}_{size}",
            compute_units=compute_units,
            num_steps=num_steps,
            size=size,
            quantize_nbits=quantize_nbits,
            auto_export=True,
            cache_compiled=True,
            out_dir=t_dir,
            seed=seed,
            reconcile_steps=reconcile_steps,
        )
        primes = [
            torch.from_numpy(np.array(Image.open(Path(t_dir) / f"prime_{k}.png")).astype(np.float32) / 255.0).permute(2, 0, 1)
            for k in range(op.num_primes)
        ]
        if "scrambled_perm" in metrics:
            print(f"Rubik's step-0 scrambled face match: perm={metrics['scrambled_perm']} | MSE {metrics['match_mse_before']:.5f} -> {metrics['match_mse_after']:.5f} | {metrics['scrambled_face_prompts']}")
        views_01 = op.derive(primes)
        return primes, views_01, float(metrics["gen_s"])

    # 2. Standalone fallback (when notebook is opened on Mac without cloning the full repo)
    import coremltools as ct
    import coremltools.optimize as cto
    from coremltools.models import CompiledMLModel
    from coremltools.models.utils import compile_model

    if not (platform.system() == "Darwin" and platform.machine() in ("arm64", "arm64e")):
        raise RuntimeError("run_apple_npu requires macOS Apple Silicon (arm64).")

    class _ANEAttnProc:
        def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, temb=None, *a, **kw):
            residual = hidden_states
            if attn.spatial_norm is not None:
                hidden_states = attn.spatial_norm(hidden_states, temb)
            ndim = hidden_states.ndim
            if ndim == 4:
                b, c, h, w = hidden_states.shape
                hidden_states = hidden_states.flatten(2).transpose(1, 2)
            if attention_mask is not None:
                bs, sl, _ = (hidden_states.shape if encoder_hidden_states is None else encoder_hidden_states.shape)
                attention_mask = attn.prepare_attention_mask(attention_mask, sl, bs).view(bs, attn.heads, 1, -1)
            if attn.group_norm is not None:
                hidden_states = attn.group_norm(hidden_states.transpose(1, 2)).transpose(1, 2)
            hs_4d = hidden_states.transpose(1, 2).unsqueeze(2)
            if encoder_hidden_states is None:
                enc_4d = hs_4d
            else:
                if attn.norm_cross is not None:
                    encoder_hidden_states = attn.norm_encoder_hidden_states(encoder_hidden_states)
                enc_4d = encoder_hidden_states.transpose(1, 2).unsqueeze(2)
            q_4d = F.conv2d(hs_4d, attn.to_q.weight[:, :, None, None], attn.to_q.bias)
            k_4d = F.conv2d(enc_4d, attn.to_k.weight[:, :, None, None], attn.to_k.bias)
            v_4d = F.conv2d(enc_4d, attn.to_v.weight[:, :, None, None], attn.to_v.bias)
            hd = int(attn.to_q.out_features // attn.heads)
            outs = []
            for h_idx, (qh, kh, vh) in enumerate(zip(q_4d.split(hd, 1), k_4d.split(hd, 1), v_4d.split(hd, 1))):
                sc = torch.matmul(qh.permute(0, 2, 3, 1), kh.permute(0, 2, 1, 3)) * attn.scale
                if attention_mask is not None:
                    sc = sc + attention_mask[:, h_idx:h_idx + 1]
                outs.append(torch.matmul(torch.softmax(sc, dim=-1), vh.permute(0, 2, 3, 1)).permute(0, 3, 1, 2))
            out_4d = F.conv2d(torch.cat(outs, 1), attn.to_out[0].weight[:, :, None, None], attn.to_out[0].bias)
            hs = out_4d.squeeze(2).transpose(1, 2)
            if len(attn.to_out) > 1:
                hs = attn.to_out[1](hs)
            if ndim == 4:
                hs = hs.transpose(-1, -2).reshape(b, c, h, w)
            if attn.residual_connection:
                hs = hs + residual
            return hs / attn.rescale_output_factor

    # NumPy >= 2.0 scalar cast compat for coremltools TorchScript frontend
    from coremltools.converters.mil import Builder as mb
    from coremltools.converters.mil.frontend.torch import ops as torch_ops
    from coremltools.converters.mil.frontend.torch.ops import _get_inputs
    if not getattr(torch_ops, "_ambigram_cast_patched", False):
        _orig_cast = getattr(torch_ops, "_cast", None)
        if _orig_cast is not None:
            def _safe_cast(ctx, node, dt, dt_name):
                try:
                    return _orig_cast(ctx, node, dt, dt_name)
                except TypeError:
                    x = _get_inputs(ctx, node, expected=1)[0]
                    if x.val is not None and np.asarray(x.val).size == 1:
                        ctx.add(mb.const(val=dt(np.asarray(x.val).reshape(()).item()), name=node.name))
                        return
                    raise
            torch_ops._cast = _safe_cast
            torch_ops._ambigram_cast_patched = True

    bundle_dir = Path(f"./coreml_models/{model_key}_{size}")
    if not (bundle_dir / "meta.json").exists():
        if model_key != "sd_turbo":
            raise RuntimeError(f"Standalone fallback auto-exports sd_turbo; clone the repo for {model_key} export.")
        from diffusers import StableDiffusionPipeline
        print(f"Exporting ANE-optimized Core ML sd_turbo ({size}px, {quantize_nbits}-bit) to {bundle_dir}...")
        bundle_dir.mkdir(parents=True, exist_ok=True)
        pipe = StableDiffusionPipeline.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float32, safety_checker=None, requires_safety_checker=False)
        for sub in pipe.unet.modules():
            if hasattr(sub, "set_processor") and hasattr(sub, "to_q") and hasattr(sub, "to_out") and hasattr(sub, "scale"):
                sub.set_processor(_ANEAttnProc())
        lat_h = lat_w = size // 8
        seq_len = pipe.text_encoder.config.max_position_embeddings
        txt_dim = pipe.text_encoder.config.hidden_size

        class _U(nn.Module):
            def __init__(self, u): super().__init__(); self.u = u
            def forward(self, s, t, e): return self.u(s, t, encoder_hidden_states=e, return_dict=False)[0]
        class _VD(nn.Module):
            def __init__(self, v): super().__init__(); self.v = v
            def forward(self, z): return self.v.decode(z, return_dict=False)[0]
        class _VE(nn.Module):
            def __init__(self, v): super().__init__(); self.v = v
            def forward(self, im): return self.v.encode(im).latent_dist.mean
        class _TE(nn.Module):
            def __init__(self, te): super().__init__(); self.te = te
            def forward(self, ids): return self.te(ids, return_dict=False)[0]

        def _conv(mod, ex, ins, outs, p, nbits):
            with torch.no_grad():
                tr = torch.jit.trace(mod.eval(), ex, check_trace=False)
            ml = ct.convert(tr, convert_to="mlprogram", inputs=ins, outputs=outs, compute_precision=ct.precision.FLOAT16,
                            minimum_deployment_target=ct.target.macOS14 if (nbits and nbits < 16) else ct.target.macOS13)
            if nbits and nbits < 16:
                cfg = cto.coreml.OptimizationConfig(global_config=cto.coreml.OpPalettizerConfig(mode="kmeans", nbits=int(nbits), weight_threshold=512))
                ml = cto.coreml.palettize_weights(ml, cfg)
            ml.save(str(p))

        s_ex = torch.randn(1, 4, lat_h, lat_w)
        t_ex = torch.tensor([999.0], dtype=torch.float32)
        e_ex = torch.randn(1, seq_len, txt_dim)
        _conv(_U(pipe.unet), (s_ex, t_ex, e_ex),
              [ct.TensorType("sample", (1, 4, lat_h, lat_w), np.float16), ct.TensorType("timestep", (1,), np.float16), ct.TensorType("encoder_hidden_states", (1, seq_len, txt_dim), np.float16)],
              [ct.TensorType("noise_pred", dtype=np.float16)], bundle_dir / "unet.mlpackage", quantize_nbits)
        _conv(_VD(pipe.vae), (s_ex,), [ct.TensorType("z", (1, 4, lat_h, lat_w), np.float16)], [ct.TensorType("image", dtype=np.float16)], bundle_dir / "vae_decoder.mlpackage", None)
        _conv(_VE(pipe.vae), (torch.randn(1, 3, size, size),), [ct.TensorType("image", (1, 3, size, size), np.float16)], [ct.TensorType("latent", dtype=np.float16)], bundle_dir / "vae_encoder.mlpackage", None)
        _conv(_TE(pipe.text_encoder), (torch.zeros((1, seq_len), dtype=torch.int32),), [ct.TensorType("input_ids", (1, seq_len), np.int32)], [ct.TensorType("last_hidden_state", dtype=np.float16)], bundle_dir / "text_encoder.mlpackage", quantize_nbits)
        (bundle_dir / "meta.json").write_text(json.dumps({"model_key": "sd_turbo", "model_id": "stabilityai/sd-turbo", "size": size, "latent_h": lat_h, "latent_w": lat_w, "scaling_factor": float(pipe.vae.config.scaling_factor)}))

    cu = getattr(ct.ComputeUnit, compute_units)
    def _load(name):
        pkg = bundle_dir / f"{name}.mlpackage"
        comp = bundle_dir / f"{name}.mlmodelc"
        if not comp.exists():
            compile_model(str(pkg), str(comp))
        return CompiledMLModel(str(comp), compute_units=cu)

    cache_k = f"ane_{model_key}_{size}_{compute_units}"
    if cache_k not in _PIPE_CACHE:
        _clear_other_pipes(cache_k)
        _PIPE_CACHE[cache_k] = (_load("unet"), _load("vae_decoder"), _load("vae_encoder"), _load("text_encoder"), json.loads((bundle_dir / "meta.json").read_text()))
    u_cm, vd_cm, ve_cm, te_cm, meta = _PIPE_CACHE[cache_k]

    from diffusers import EulerDiscreteScheduler
    from transformers import CLIPTokenizer
    tok = CLIPTokenizer.from_pretrained(meta["model_id"], subfolder="tokenizer")
    sched = EulerDiscreteScheduler.from_pretrained(meta["model_id"], subfolder="scheduler", timestep_spacing="trailing")
    ids = tok(prompts, padding="max_length", max_length=tok.model_max_length, truncation=True, return_tensors="np").input_ids.astype(np.int32)

    pixel_reconcile = (not op.is_permutation) or (not op.equivariant_to_latent)
    reconcile_dev = _get_device()
    t0 = time.perf_counter()
    emb = np.concatenate([te_cm.predict({"input_ids": ids[k:k + 1]})["last_hidden_state"] for k in range(op.num_views)], axis=0).astype(np.float16)
    sched.set_timesteps(num_steps, device="cpu")
    sigmas = sched.sigmas.numpy().astype(np.float32)
    sf = float(meta["scaling_factor"])
    gen = torch.Generator(device="cpu").manual_seed(seed)
    lat = torch.randn(op.num_primes if (op.is_permutation and not pixel_reconcile) else op.num_views, 4, meta["latent_h"], meta["latent_w"], generator=gen)
    if op.is_permutation and not pixel_reconcile:
        lat = op.derive([lat[k] for k in range(op.num_primes)])
    lat = lat * float(sched.init_noise_sigma)

    warm_logits, primes, last_rec = None, None, None
    for i, t in enumerate(sched.timesteps):
        sc = float(sched.scale_model_input(torch.ones(1), t).item())
        t_np = np.array([float(t.item())], dtype=np.float16)
        np_all = torch.cat([torch.from_numpy(u_cm.predict({"sample": (lat[v:v + 1].numpy() * sc).astype(np.float16), "timestep": t_np, "encoder_hidden_states": emb[v:v + 1]})["noise_pred"].astype(np.float32)) for v in range(op.num_views)], dim=0)
        sig, sig_n = float(sigmas[i]), float(sigmas[i + 1])
        x0 = lat - sig * np_all
        if op.is_permutation and not pixel_reconcile:
            lat = sched.step(op.invert_velocities(np_all), t, lat, return_dict=False)[0]
        else:
            z0 = (x0 / sf).numpy().astype(np.float16)
            imgs = ((torch.cat([torch.from_numpy(vd_cm.predict({"z": z0[v:v + 1]})["image"].astype(np.float32)) for v in range(op.num_views)], 0) + 1.0) * 0.5).clamp(0.0, 1.0).to(reconcile_dev)
            if i == 0 and isinstance(op, RubiksCubeOperator):
                _reorder_rubiks_scrambled(op, imgs, prompts, lat, emb)
            if op.is_permutation:
                rec = op.invert_and_average(imgs)
                primes = [rec[k].cpu() for k in range(op.num_primes)]
            elif isinstance(op, RotationOverlayOperator):
                rec, pd, warm_logits = op.reconcile(imgs, init_logits=warm_logits, steps=reconcile_steps)
                primes = [p.cpu() for p in pd]
            else:
                rec, pd = op.reconcile(imgs)
                primes = [p.cpu() for p in pd]
            last_rec = rec.cpu()
            if i + 1 < len(sched.timesteps) or sig > 1e-4:
                r_np = (2.0 * last_rec - 1.0).numpy().astype(np.float16)
                r_lat = torch.cat([torch.from_numpy(ve_cm.predict({"image": r_np[v:v + 1]})["latent"].astype(np.float32)) for v in range(op.num_views)], 0) * sf
                lat = lat + (sig_n - sig) * ((lat - r_lat) / sig) if sig > 1e-4 else r_lat
    if op.is_permutation and not pixel_reconcile:
        zf = (lat / sf).numpy().astype(np.float16)
        dec = ((torch.cat([torch.from_numpy(vd_cm.predict({"z": zf[v:v + 1]})["image"].astype(np.float32)) for v in range(op.num_views)], 0) + 1.0) * 0.5).clamp(0.0, 1.0)
        primes = [dec[k] for k in range(op.num_primes)]
    elif op.is_permutation:
        primes = [last_rec[k] for k in range(op.num_primes)]
    return primes, op.derive(primes), time.perf_counter() - t0

print("Runners ready.")

In [ ]:
#@title 4. Generate Illusion
TASK = "flip"  #@param ["flip", "twisting_squares", "rubiks_cube", "rotation_overlay", "hidden_overlay"]
METHOD = "auto"  #@param ["auto", "apple_npu", "flux2_klein", "sana_sprint", "iris3b_turbo", "fast_sds"]
APPLE_NPU_MODEL = "sd_turbo"  #@param ["sd_turbo", "sdxl_turbo", "sana_sprint"]
APPLE_NPU_QUANT_BITS = 6  #@param [6, 8, 16]
SEED = 42  #@param {type:"integer"}
STEPS = 0  #@param {type:"integer"}  # 0 = auto
SIZE = 0  #@param {type:"integer"}   # 0 = auto (divisible by 3 for rubiks_cube)
SDS_ITERS = 350  #@param {type:"integer"}
PROMPTS_OVERRIDE = ""  #@param {type:"string"}

DEFAULT_PROMPTS = {
    "flip": [
        "a pencil sketch of a giraffe's head",
        "a pencil sketch of a penguin",
    ],
    "twisting_squares": [
        "Hatsune Miku",
        "A dog's face photograph 4k",
    ],
    "rubiks_cube": [
        "vibrant oil painting of a sunflower field under blue sky",
        "vibrant oil painting of a red fox in a green forest",
        "vibrant oil painting of a majestic lion portrait",
        "vibrant oil painting of a colorful parrot in tropical leaves",
        "vibrant oil painting of a sailing ship on blue ocean waves",
        "vibrant oil painting of a snow-capped mountain at sunset",
        "vibrant oil painting of a golden retriever dog portrait",
        "vibrant oil painting of an owl perched in autumn woods",
        "vibrant oil painting of a Bengal tiger portrait",
        "vibrant oil painting of a lighthouse by crashing ocean waves",
        "vibrant oil painting of a vintage red steam train",
        "vibrant oil painting of a butterfly on colorful flowers",
    ],
    "rotation_overlay": [
        "Hatsune Miku anime portrait, vibrant contrast",
        "Cute green tree frog portrait",
        "Red lipstick tube, studio lighting",
        "Egyptian pyramids and sphinx at sunset",
    ],
    "hidden_overlay": [
        "oil painting of a red fox in a forest",
        "oil painting of a snowy mountain cabin",
        "oil painting of a sailing ship on the ocean",
        "oil painting of a sunflower field at sunset",
        "oil painting of a majestic lion portrait",
    ],
}

prompts = [p.strip() for p in PROMPTS_OVERRIDE.split("|") if p.strip()] if PROMPTS_OVERRIDE.strip() else DEFAULT_PROMPTS[TASK]

active_method = ("apple_npu" if BACKEND_MODE == "apple_npu" else "flux2_klein") if METHOD == "auto" else METHOD
if active_method == "apple_npu" and BACKEND_MODE != "apple_npu":
    raise RuntimeError(f"METHOD='apple_npu' requires macOS 13+ Apple Silicon with coremltools (current BACKEND_MODE={BACKEND_MODE!r}).")

if SIZE <= 0:
    if active_method == "apple_npu":
        if TASK == "rubiks_cube":
            size = 768 if APPLE_NPU_MODEL == "sana_sprint" else (576 if APPLE_NPU_MODEL == "sdxl_turbo" else 480)
        else:
            size = 1024 if APPLE_NPU_MODEL == "sana_sprint" else 512
    elif TASK == "rubiks_cube":
        size = 768 if active_method in ("sana_sprint", "iris3b_turbo") else (576 if active_method == "flux2_klein" else 480)
    else:
        size = 1024 if active_method in ("sana_sprint", "iris3b_turbo") else 512
else:
    size = SIZE

print(f"Backend={BACKEND_MODE} | task={TASK} | method={active_method} | size={size} | seed={SEED}")
print("Prompts:", prompts)

if active_method == "apple_npu":
    steps = STEPS if STEPS > 0 else (2 if (APPLE_NPU_MODEL == "sana_sprint" and TASK != "rubiks_cube") else 4)
    qbits = APPLE_NPU_QUANT_BITS if APPLE_NPU_QUANT_BITS < 16 else None
    primes, views_01, elapsed_s = run_apple_npu(
        TASK, prompts, model_key=APPLE_NPU_MODEL, num_steps=steps, size=size, quantize_nbits=qbits, seed=SEED
    )
elif active_method == "flux2_klein":
    steps = STEPS if STEPS > 0 else (6 if TASK == "rubiks_cube" else 4)
    primes, views_01, elapsed_s = run_flux2_klein(TASK, prompts, num_steps=steps, size=size, seed=SEED)
elif active_method == "sana_sprint":
    steps = STEPS if STEPS > 0 else (4 if TASK in ("twisting_squares", "rubiks_cube") else 2)
    primes, views_01, elapsed_s = run_sana_sprint(TASK, prompts, num_steps=steps, size=size, seed=SEED)
elif active_method == "iris3b_turbo":
    primes, views_01, elapsed_s = run_iris3b_turbo(TASK, prompts, size=size, seed=SEED)
elif active_method == "fast_sds":
    primes, views_01, elapsed_s = run_fast_sds(TASK, prompts, sds_iters=SDS_ITERS, size=size, seed=SEED)
else:
    raise ValueError(f"Unknown method: {active_method}")

out_dir = Path(f"outputs/{TASK}/{active_method}")
out_dir.mkdir(parents=True, exist_ok=True)

for idx, p in enumerate(primes):
    tensor_to_pil(p).save(out_dir / f"prime_{idx}.png")
view_pils = [tensor_to_pil(v) for v in views_01]
for idx, v in enumerate(view_pils):
    v.save(out_dir / f"view_{idx}.png")
grid_pil = make_grid(view_pils)
grid_pil.save(out_dir / "final_grid.png")

print(f"Done in {elapsed_s:.3f}s! Saved to {out_dir}")
display(grid_pil)

In [ ]:
#@title 5. Inspect Individual Prime Sheets & Views (or Zip for Download)
import shutil

print("Individual Derived Views:")
for p in sorted(out_dir.glob("view_*.png")):
    print(p.name)
    display(Image.open(p))

print("Physical Prime Layers (to print / rotate / overlay):")
for p in sorted(out_dir.glob("prime_*.png")):
    print(p.name)
    display(Image.open(p))

MAKE_ZIP = False  #@param {type:"boolean"}
if MAKE_ZIP:
    zip_path = shutil.make_archive(f"{TASK}_{active_method}_results", "zip", out_dir)
    print("Created archive:", zip_path)
    try:
        from google.colab import files
        files.download(zip_path)
    except ImportError:
        pass